# Data Cleaning

This notebook aligns all six raw datasets to a common weekly time index and assembles them into a single master dataframe (`master_weekly.csv`). Each source is resampled or forward-filled from its native frequency (daily, monthly, or annual) to weekly, then joined on the date index.

## Dependencies

Re-runs `data_pulling.ipynb` to bring all raw variables (`yf_data`, `cpi`, `unemployment`, `interest_rate`, `trends_df`, `fluview_df`, `wonder_df`, `cms_df`) into the current namespace.

In [20]:
%run data_pulling.ipynb

[*********************100%***********************]  4 of 4 completed


2010-01-01    217.488
2010-02-01    217.281
2010-03-01    217.353
2010-04-01    217.403
2010-05-01    217.290
2010-06-01    217.199
2010-07-01    217.605
2010-08-01    217.923
2010-09-01    218.275
2010-10-01    219.035
dtype: float64
(196,)
Done: 2010-01-01 2013-12-31
Done: 2014-01-01 2017-12-31
Done: 2018-01-01 2021-12-31
Done: 2022-01-01 2024-01-01
(732, 3)
            flu symptoms  fever  shortness of breath
date                                                
2009-12-27            20     41                    3
2010-01-03            16     43                    3
2010-01-10            14     46                    2
2010-01-17            13     44                    3
2010-01-24            11     48                    3
(731, 16)
  release_date region   issue  epiweek  lag  num_ili  num_patients  \
0   2013-12-31    nat  201352   201001  207    14299        721138   
1   2013-12-31    nat  201352   201002  206    14088        770895   
2   2013-12-31    nat  201352   201003  205   

## Resampling to Weekly Frequency

Each dataset arrives at a different native cadence. This section converts all of them to Sunday-ended ISO weeks (`W` offset) so they share a common index for joining.

### Yahoo Finance — Weekly Closing Prices and Volatility

Daily prices are resampled by taking the last closing price of each week. Weekly volatility is computed as the standard deviation of daily log-returns within each week.

In [21]:
returns = yf_data['Close'].pct_change()

weekly_close = yf_data['Close'].resample('W').last()
weekly_vol = returns.resample('W').std()

weekly_vol.columns = [f"{col}_vol" for col in weekly_vol.columns]

print(weekly_close.shape)
print(weekly_close.head())
print(weekly_vol.head())

(730, 4)
Ticker           KIE        PJP        XBI        XLV
2010-01-10  9.090796  15.240718  17.882545  24.113842
2010-01-17  9.083311  15.451389  17.709146  24.447704
2010-01-24  8.753912  15.070560  17.683453  24.022779
2010-01-31  8.801326  15.086777  17.750891  23.734453
2010-02-07  8.689034  14.859903  17.635288  23.393009
             KIE_vol   PJP_vol   XBI_vol   XLV_vol
2010-01-10  0.008362  0.003628  0.005640  0.008323
2010-01-17  0.009673  0.010266  0.013155  0.009322
2010-01-24  0.018905  0.019470  0.016492  0.019807
2010-01-31  0.011472  0.007059  0.010714  0.005937
2010-02-07  0.022650  0.017009  0.027530  0.017616


The result is 730 weekly observations across 4 ETFs (2010-01-10 to 2023-12-31), each paired with a corresponding weekly volatility measure — the primary target variable for the analysis.

### FRED — Weekly Macroeconomic Indicators

FRED series are monthly so each week within a month carries the same value (forward-fill). Combining all three series into one dataframe simplifies the downstream join.

In [22]:
cpi_weekly = cpi.resample('W').ffill()
unemployment_weekly = unemployment.resample('W').ffill()
interest_rate_weekly = interest_rate.resample('W').ffill()

fred_weekly = pd.DataFrame({
    'cpi': cpi_weekly,
    'unemployment': unemployment_weekly,
    'interest_rate': interest_rate_weekly
})

print(fred_weekly.shape)
print(fred_weekly.head())

(853, 3)
                cpi  unemployment  interest_rate
2010-01-03  217.488           9.8           0.11
2010-01-10  217.488           9.8           0.11
2010-01-17  217.488           9.8           0.11
2010-01-24  217.488           9.8           0.11
2010-01-31  217.488           9.8           0.11


853 weekly rows span the full FRED history available; the inner join with yfinance will trim this to the 2010–2023 window. Forward-filling is appropriate here because macro indicators are published monthly and do not change between releases.

### Google Trends — Weekly Symptom Searches

Renames raw column headers to prefixed names to avoid collisions in the master dataframe and ensures the index is a proper DatetimeIndex.

In [23]:
trends_df.index = pd.to_datetime(trends_df.index)
trends_df.columns = ['trends_flu_symptoms', 'trends_fever', 'trends_shortness_of_breath']

print(trends_df.shape)
print(trends_df.head())

(732, 3)
            trends_flu_symptoms  trends_fever  trends_shortness_of_breath
date                                                                     
2009-12-27                   20            41                           3
2010-01-03                   16            43                           3
2010-01-10                   14            46                           2
2010-01-17                   13            44                           3
2010-01-24                   11            48                           3


### CDC FluView — Weekly ILI

FluView encodes dates as epidemiological week numbers (YYYYWW format). This cell converts them to standard Monday-anchored datetimes using `strptime` with the `%Y%W%w` directive, then retains only the four columns needed for analysis.

In [24]:
import pandas as pd

# epiweek format is YYYYWW (e.g. 201001 = 2010 week 1)
# We need to convert this to a proper datetime
# pd.to_datetime with format %Y%W%w converts year+week+day (1=Monday)
fluview_df['date'] = pd.to_datetime(
    fluview_df['epiweek'].astype(str) + '1', 
    format='%Y%W%w'
)
fluview_df = fluview_df.set_index('date')

# Keep only the columns we need
fluview_weekly = fluview_df[['wili', 'ili', 'num_ili', 'num_patients']]

print(fluview_weekly.shape)
print(fluview_weekly.head())

(731, 4)
                wili       ili  num_ili  num_patients
date                                                 
2010-01-04  1.907118  1.982838    14299        721138
2010-01-11  1.867375  1.827486    14088        770895
2010-01-18  1.880723  1.926056    14757        766177
2010-01-25  1.969084  1.924947    15122        785580
2010-02-01  2.113868  2.088768    16037        767773


731 weekly ILI observations align almost exactly with the yfinance window, confirming no significant date gaps in the surveillance record.

### NCHS — Weekly Mortality (Forward-Filled from Annual)

NCHS publishes annual national death totals. Deaths are summed across all states and causes for each year, then the annual series is resampled to weekly and forward-filled so each week in a given year carries that year's total. This is an approximation — NCHS data will later be confirmed as a low-utility feature due to time-trend confounding.

In [26]:
# NCHS is annual data - convert year to datetime and ffill to weekly
# We'll aggregate deaths by year first (sum across all states and causes)
wonder_df['year'] = pd.to_datetime(wonder_df['year'], format='%Y')
wonder_df['deaths'] = pd.to_numeric(wonder_df['deaths'], errors='coerce')

# Group by year and sum total deaths
nchs_annual = wonder_df.groupby('year')['deaths'].sum()

# Resample to weekly and forward fill
nchs_weekly = nchs_annual.resample('W').ffill()

print(nchs_weekly.shape)
print(nchs_weekly.head())

(940,)
year
1999-01-03    8594450
1999-01-10    8594450
1999-01-17    8594450
1999-01-24    8594450
1999-01-31    8594450
Freq: W-SUN, Name: deaths, dtype: int64


940 weekly rows reflect the longer NCHS history (1999 onward); only 2010–2017 overlaps the study window because the CDC dataset used here cuts off at 2017.

## Data Inspection

Before merging, confirm the structure and usability of each dataset.

### CMS — Column Inventory

CMS inpatient data is cross-sectional (provider-level, not time-series), so it cannot be joined on a date index. Inspecting the columns confirms there is no year or date field to align it with the weekly master frame.

In [27]:
# CMS doesn't have a year column directly - check what we have
print(cms_df.columns.tolist())

['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt']


CMS data contains provider identifiers, DRG codes, and payment averages but no temporal dimension suitable for a weekly join. It will be excluded from the master dataframe; its role in the project is descriptive rather than predictive.

## Building Master DataFrame

Joins all time-indexed datasets on their weekly date index. An inner join on the five time-series sources ensures every row has complete data for the primary variables. NCHS is left-joined because its history ends in 2017; missing post-2017 values are forward-filled from the last known annual total.

In [34]:
# Merge everything except NCHS first with inner join
master_df = weekly_close_r.copy()
master_df.index.name = 'date'

master_df = master_df.join(weekly_vol_r, how='inner')
master_df = master_df.join(fred_weekly_r, how='inner')
master_df = master_df.join(trends_r, how='inner')
master_df = master_df.join(fluview_r, how='inner')

# Join NCHS with outer then ffill so we don't lose post-2017 data
master_df = master_df.join(nchs_r.rename('nchs_deaths'), how='left')
master_df['nchs_deaths'] = master_df['nchs_deaths'].ffill()

print(master_df.shape)
print(master_df.index.min(), "to", master_df.index.max())
print(master_df.isnull().sum())

(730, 19)
2010-01-10 00:00:00 to 2023-12-31 00:00:00
KIE                           0
PJP                           0
XBI                           0
XLV                           0
KIE_vol                       0
PJP_vol                       0
XBI_vol                       0
XLV_vol                       0
cpi                           0
unemployment                  0
interest_rate                 0
trends_flu_symptoms           0
trends_fever                  0
trends_shortness_of_breath    0
wili                          2
ili                           2
num_ili                       2
num_patients                  2
nchs_deaths                   0
dtype: int64


The master dataframe has 730 rows and 19 columns, spanning 2010-01-10 to 2023-12-31. The small number of null values in the FluView columns are at the boundary edges where the epiweek calendar does not align perfectly with the Sunday week anchor.

## Null Handling

The two missing FluView rows result from a one-week calendar misalignment at the series boundary. Forward-filling propagates the nearest valid observation, which is appropriate for a slowly-changing surveillance metric.

In [35]:
master_df[['wili', 'ili', 'num_ili', 'num_patients']] = master_df[['wili', 'ili', 'num_ili', 'num_patients']].ffill()

print(master_df.isnull().sum().sum())

0


Zero total nulls confirms the master dataframe is fully populated and ready for analysis.

## Date Range Validation

Verifies the actual date coverage of each source after resampling. The binding constraint (shortest window) determines the final master dataframe range.

In [32]:
print("yfinance:", weekly_close_r.index.min(), "to", weekly_close_r.index.max())
print("fred:", fred_weekly_r.index.min(), "to", fred_weekly_r.index.max())
print("trends:", trends_r.index.min(), "to", trends_r.index.max())
print("fluview:", fluview_r.index.min(), "to", fluview_r.index.max())
print("nchs:", nchs_r.index.min(), "to", nchs_r.index.max())

yfinance: 2010-01-10 00:00:00 to 2023-12-31 00:00:00
fred: 2010-01-03 00:00:00 to 2026-05-03 00:00:00
trends: 2009-12-27 00:00:00 to 2023-12-31 00:00:00
fluview: 2010-01-10 00:00:00 to 2024-01-07 00:00:00
nchs: 1999-01-03 00:00:00 to 2017-01-01 00:00:00


yfinance (2010–2023) is the binding constraint and sets the master date range. FRED extends further into the future and NCHS ends earlier, but both are handled correctly by the inner/left join strategy.

## Saving Processed Data

Writes the final master dataframe to `master_weekly.csv` in the working directory for use by the EDA and modeling notebooks.

In [36]:
master_df.to_csv('../data/processed/master_weekly.csv')
print("Saved.")

Saved.
